# Concurrency - doing many requests at once instead of one-by-one

In [1]:
import requests
import concurrent.futures

In [2]:
urls = [
    'http://books.toscrape.com/catalogue/page-1.html',
    'http://books.toscrape.com/catalogue/page-2.html',
    'http://books.toscrape.com/catalogue/page-3.html',
]

In [3]:
def fetch(url):
    response = requests.get(url, timeout=10)
    return url, response.status_code, len(response.text)

In [4]:
with concurrent.futures.ThreadPoolExecutor(max_workers=5) as e:
    results = e.map(fetch, urls)

for url, status, length in results:
    print(url, status, length)

http://books.toscrape.com/catalogue/page-1.html 200 50469
http://books.toscrape.com/catalogue/page-2.html 200 50877
http://books.toscrape.com/catalogue/page-3.html 200 51374


# Scrapy

In [5]:
pip install scrapy

   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   -------- ------------------------------- 0.8/3.8 MB 4.8 MB/s eta 0:00:01
   ------------------- -------------------- 1.8/3.8 MB 5.3 MB/s eta 0:00:01
   ------------------------------ --------- 2.9/3.8 MB 5.4 MB/s eta 0:00:01
   ---------------------------------------- 3.8/3.8 MB 5.2 MB/s  0:00:00
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ------------- -------------------------- 1.3/4.0 MB 5.6 MB/s eta 0:00:01
   ---------------------------- ----------- 2.9/4.0 MB 6.7 MB/s eta 0:00:01
   ---------------------------------------- 4.0/4.0 MB 6.7 MB/s  0:00:00
   ---------------------------------------- 0.0/3.2 MB ? eta -:--:--
   ---------------------- ----------------- 1.8/3.2 MB 7.7 MB/s eta 0:00:01
   ---------------------------------------- 3.2/3.2 MB 7.9 MB/s  0:00:00

   - --------------------------------------  1/23 [zope-interface]
   - --------------------------------------  1/23 

In [6]:
import scrapy

class BooksSpider(scrapy.Spider):
    name = 'books'
    start_urls = ['http://books.toscrape.com/']

    def parse(self, response):
        for book in response.css('article.product_pod'):
            yield {
                'title': book.css('h3 a::attr(title)').get(),
                'price': book.css('p.price_color::text').get(),
            }

        next_page = response.css('li.next a::attr(href)').get()

        if next_page:
            yield response.follow(next_page, callback=self.parse)